In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


# ==========================================
# 1. IMAGE PREPROCESSING
# ==========================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])


# ==========================================
# 2. DATASET
# ==========================================

trainset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

testset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# ==========================================
# 3. DATALOADERS
# ==========================================

trainloader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)


# ==========================================
# 4. CNN MODEL
# ==========================================

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(

            # 3 → 32 channels
            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),


            # 32 → 64 channels
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),


            # 64 → 128 channels
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )


        self.fc_layers = nn.Sequential(

            # 128 × 4 × 4 = 2048
            nn.Linear(
                4 * 4 * 128,
                256
            ),

            nn.ReLU(),

            # 10 CIFAR-10 classes
            nn.Linear(
                256,
                10
            )
        )


    def forward(self, x):

        # Feature extraction
        x = self.conv_layers(x)

        # Flatten
        x = x.view(
            x.size(0),
            -1
        )

        # Classification
        x = self.fc_layers(x)

        return x


# ==========================================
# 5. CREATE MODEL
# ==========================================

model = CNN()


# ==========================================
# 6. LOSS + OPTIMIZER
# ==========================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# ==========================================
# 7. TRAINING
# ==========================================

epochs = 10

for epoch in range(epochs):

    model.train()

    epoch_training_loss = 0.0

    for images, labels in trainloader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        epoch_training_loss += loss.item()


    average_loss = (
        epoch_training_loss /
        len(trainloader)
    )

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )


# ==========================================
# 8. EVALUATION
# ==========================================

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():

    for images, labels in testloader:

        # Forward pass
        outputs = model(images)

        # Get class with highest score
        _, predicted = torch.max(
            outputs,
            1
        )

        # Count correct predictions
        correct_labels += (
            predicted == labels
        ).sum().item()

        # Count total images
        total_labels += labels.size(0)


accuracy = correct_labels / total_labels

print(
    f"Accuracy = {accuracy * 100:.2f}%"
)

100%|██████████| 170M/170M [24:32<00:00, 116kB/s]    


Epoch 1/10, Loss: 1.3863
Epoch 2/10, Loss: 0.9603
Epoch 3/10, Loss: 0.7595
Epoch 4/10, Loss: 0.6368
Epoch 5/10, Loss: 0.5282
Epoch 6/10, Loss: 0.4358
Epoch 7/10, Loss: 0.3569
Epoch 8/10, Loss: 0.2789
Epoch 9/10, Loss: 0.2166
Epoch 10/10, Loss: 0.1683
Accuracy = 74.93%
